In [6]:
import xarray as xr
import datetime
import numpy as np
import pandas as pd
from dask import persist
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'scripts' / 'utils.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.paths import ai_output_root, aifs_root, aurora_root, ifs_root
from scripts import utils


# CONFIGURATION SECTION

In [7]:
# Name of the storm
storm_name = 'Claudia'  # Options: 'Ciaran', 'Claudia'
models = ['fcnv2', 'pangu', 'aurora', 'aifs']
factual_models = ['fcnv2', 'pangu', 'ifs', 'aurora', 'aifs']
ifs_init_times = None

# Wind level used for the wind diagnostics
use_850_winds = True  # False -> 10 m winds, True -> 850 hPa winds
wind_level_label = '850 hPa' if use_850_winds else '10 m'
wind_speed_label = f'wind speed at {wind_level_label} [m/s]'
accum_wind_speed_label = f'Accumulated maximum wind speed at {wind_level_label} [m/s]'

EARTH2MIP_FORECAST_ROOT = str(ai_output_root())
AURORA_FORECAST_ROOT = str(aurora_root() / 'fcst')
AIFS_FORECAST_ROOT = str(aifs_root() / 'fcst')
IFS_FORECAST_ROOT = str(ifs_root())
PGW_EXPERIMENT = 'PGW_multimodel_v1'
PGW_EXTENSION = f'{PGW_EXPERIMENT}_'

# Main area of interest (plotting) and for averaging
if storm_name == 'Ciaran':
    latS, latN = 34, 58
    lonW, lonE = -20, 6
    box_latS, box_latN = 28, 48
    box_lonW, box_lonE = -15, -3
    validation_geometry = 'circle'
    validation_radius_m = 800_000
    t0_i = datetime.datetime(2023, 10, 27, 0)
    t0_f = datetime.datetime(2023, 11, 2, 0)
    delta_h = 6
    num_steps = 4 * 15
    time_slice = slice('2023-10-27', '2023-11-02')
    init_time_min = datetime.datetime(2023, 10, 27, 0)
    lead_time_range = np.arange(0, 10 * 24, 6)
    date = '2023-11-02-00'
elif storm_name == 'Claudia':
    latS, latN = 25, 55
    lonW, lonE = -25, 10
    box_latS, box_latN = 28, 48
    box_lonW, box_lonE = -15, -3
    validation_geometry = 'box'
    validation_radius_m = None
    t0_i = datetime.datetime(2025, 11, 7, 0)
    t0_f = datetime.datetime(2025, 11, 14, 0)
    delta_h = 6
    num_steps = 4 * 15
    time_slice = slice('2025-11-07', '2025-11-14')
    init_time_min = datetime.datetime(2025, 11, 7, 0)
    lead_time_range = np.arange(0, 10 * 24, 6)
    date = '2025-11-13-12'
else:
    raise ValueError(f'Unsupported storm_name: {storm_name}')


In [8]:
init_times = pd.date_range(t0_i, t0_f, freq=f"{delta_h}h").to_pydatetime().tolist()


In [9]:
import numpy as np
import metpy.calc as mpcalc
from metpy.units import units

def relative_humidity_to_specific_humidity(relative_humidity, temperature, pressure):
    """
    Convert relative humidity (%) to specific humidity (kg/kg).
    
    Parameters:
    -----------
    relative_humidity : float or array-like
        Relative humidity in percent (%).
    temperature : float or array-like
        Temperature in degrees Celsius (°C).
    pressure : float or array-like
        Pressure in hectopascals (hPa).
    
    Returns:
    --------
    specific_humidity : float or array-like
        Specific humidity in kilograms per kilogram (kg/kg).
    """
    # Convert inputs to MetPy units
    rh = relative_humidity * units.percent
    t = temperature * units.kelvin
    p = pressure * units.hPa
    
    # Calculate mixing ratio from RH, T, and P
    mixing_ratio = mpcalc.mixing_ratio_from_relative_humidity(p, t, rh)
    
    # Convert mixing ratio to specific humidity
    specific_humidity = mpcalc.specific_humidity_from_mixing_ratio(mixing_ratio)

    # Strip units and return as xarray.DataArray
    return xr.DataArray(
        data=specific_humidity.values,
        dims=relative_humidity.dims,
        coords=relative_humidity.coords,
        attrs={'units': 'kg/kg', 'long_name': 'Specific humidity'}
    )

def get_wind_speed_levels(storm_name, level='surface'):
    """Return default wind-speed contour levels for plotting.

    Parameters
    ----------
    storm_name : str
        Name of the storm (used to pick tailored contour ranges).
    level : {'surface', 'aloft'}
        'surface' returns 10 m wind ranges, 'aloft' returns 850/950 hPa ranges.
    """
    if storm_name == 'Ciaran':
        surface = np.arange(18, 33, 2)
        aloft = np.arange(30, 46, 2)
    elif storm_name == 'Eunice':
        surface = np.arange(14, 29, 2)
        aloft = np.arange(22, 40, 2)
    else:
        surface = np.arange(15, 35, 2)
        aloft = np.arange(24, 40, 2)

    return aloft if level == 'aloft' else surface


In [10]:
# Load the factual forecasts for the AI models and the optional IFS reference

data = {}
for model in factual_models:
    if model in ['fcnv2', 'pangu']:
        root = EARTH2MIP_FORECAST_ROOT
        u_var = 'u850' if use_850_winds else 'u10m'
        v_var = 'v850' if use_850_winds else 'v10m'
        data[model] = {
            'u': utils.load_data(u_var, init_times, root, extension='', model=model),
            'v': utils.load_data(v_var, init_times, root, extension='', model=model),
            'z': utils.load_data('msl', init_times, root, extension='', model=model),
        }
        if model == 'pangu':
            data[model]['q'] = utils.load_data('q850', init_times, root, extension='', model=model)
        else:
            r = utils.load_data('r850', init_times, root, extension='', model='fcnv2')
            t = utils.load_data('t850', init_times, root, extension='', model='fcnv2')
            data[model]['q'] = relative_humidity_to_specific_humidity(r, t, 850)
    elif model == 'aurora':
        root = AURORA_FORECAST_ROOT
        level = 850 if use_850_winds else None
        u_var = 'u' if use_850_winds else '10u'
        v_var = 'v' if use_850_winds else '10v'
        data[model] = {
            'u': utils.load_data(u_var, init_times, root, level=level, extension='', model=model),
            'v': utils.load_data(v_var, init_times, root, level=level, extension='', model=model),
            'z': utils.load_data('msl', init_times, root, extension='', model=model),
            'q': utils.load_data('q', init_times, root, level=850, extension='', model=model),
        }
    elif model == 'aifs':
        root = AIFS_FORECAST_ROOT
        u_var = 'u_850' if use_850_winds else '10u'
        v_var = 'v_850' if use_850_winds else '10v'
        data[model] = {
            'u': utils.load_data(u_var, init_times, root, extension='', model=model),
            'v': utils.load_data(v_var, init_times, root, extension='', model=model),
            'z': utils.load_data('msl', init_times, root, extension='', model=model),
            'q': utils.load_data('q_850', init_times, root, extension='', model=model),
        }
    elif model == 'ifs':
        ifs_init_times = pd.date_range(t0_i, t0_f, freq='12h').to_pydatetime().tolist()
        level = 850 if use_850_winds else None
        data[model] = {
            'u': utils.load_data('u', ifs_init_times, IFS_FORECAST_ROOT, level=level, extension='', model=model),
            'v': utils.load_data('v', ifs_init_times, IFS_FORECAST_ROOT, level=level, extension='', model=model),
            'z': utils.load_data('msl', ifs_init_times, IFS_FORECAST_ROOT, extension='', model=model),
            'q': utils.load_data('q', ifs_init_times, IFS_FORECAST_ROOT, level=850, extension='', model=model),
        }


Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110700/u850_fcnv2_2025110700.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110706/u850_fcnv2_2025110706.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110712/u850_fcnv2_2025110712.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110718/u850_fcnv2_2025110718.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110800/u850_fcnv2_2025110800.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110806/u850_fcnv2_2025110806.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110812/u850_fcnv2_2025110812.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110818/u850_fcnv2_2025110818.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110900/u850_fcnv2_2025110900.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110906/u850_fcnv2_2025110906.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110912/u850_fcnv2_2025110912.nc
Loading: /home/bernatj/Data/ai-forecasts/fcst/2025110918/u850_fcnv2_2025110918.nc
Loading: /home/b

In [ ]:
# Area-sliced data structure

lat_slice = slice(latN, latS)
lon_slice = slice(lonW, lonE)

persist_items = []
for model in factual_models:
    d = data[model]
    print(model)
    data[model]['area'] = {
        'u': utils.subset_region(d['u'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
        'v': utils.subset_region(d['v'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
        'z': utils.subset_region(d['z'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
        'q': utils.subset_region(d['q'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
    }
    for key, value in data[model]['area'].items():
        persist_items.append((model, key, value))

persisted_vars = persist(*(value for _, _, value in persist_items))
for (model, key, _), value in zip(persist_items, persisted_vars):
    data[model]['area'][key] = value


fcnv2
pangu
ifs
aurora
aifs


In [ ]:
for model in factual_models:
    area_data = data[model]['area']

    u = area_data['u']
    v = area_data['v']
    wsp = np.sqrt(u**2 + v**2)
    time_range = wsp.time.sel(time=time_slice)

    wsp_lag = utils.from_init_time_to_leadtime(wsp, init_time_min, lead_time_range, time_range).load()
    v_lag = utils.from_init_time_to_leadtime(v, init_time_min, lead_time_range, time_range).load()
    u_lag = utils.from_init_time_to_leadtime(u, init_time_min, lead_time_range, time_range).load()
    z_lag = utils.from_init_time_to_leadtime(area_data['z'], init_time_min, lead_time_range, time_range).load()
    q_lag = utils.from_init_time_to_leadtime(area_data['q'], init_time_min, lead_time_range, time_range).load()

    data[model]['leadtime'] = {
        'wsp': wsp_lag,
        'u': u_lag,
        'v': v_lag,
        'z': z_lag,
        'q': q_lag,
    }


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import numpy as np
import matplotlib.patches as patches

leadtime = 3 * 24  # hours
models_to_plot = ['fcnv2', 'pangu', 'ifs', 'aurora', 'aifs']

f, axes = plt.subplots(2, 3, figsize=(10, 7), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=0.0)))
axes = axes.flatten()

levels_q850 = np.arange(4, 9.1, 1)
levels_wsp = get_wind_speed_levels(storm_name, level='aloft' if use_850_winds else 'surface')
slp_levels = np.arange(950, 1030, 5)
factor_slp = 0.01
letter = ['a', 'b', 'c', 'd', 'e', 'f']

for ax in axes:
    ax.coastlines(lw=1.0, color='green', zorder=105)
    ax.set_extent([lonW, lonE, latS, latN])

wsp_era = data['fcnv2']['leadtime']['wsp'].sel(lead_time=0, time=date)
q_era = data['fcnv2']['leadtime']['q'].sel(lead_time=0, time=date)
slp_era = data['fcnv2']['leadtime']['z'].sel(lead_time=0, time=date)
cf0 = axes[0].contourf(wsp_era.lon, wsp_era.lat, wsp_era, cmap='plasma_r', levels=levels_wsp, extend='max', alpha=0.8, zorder=102)
cf1 = axes[0].contourf(q_era.lon, q_era.lat, q_era * 1000, cmap='PuBuGn', levels=levels_q850, extend='max', alpha=0.8, zorder=101)
con0 = axes[0].contour(slp_era.lon, slp_era.lat, slp_era * factor_slp, colors='0.2', linestyles='solid', linewidths=0.8, levels=slp_levels, zorder=106)
axes[0].contour(q_era.lon, q_era.lat, q_era * 1000, colors='b', linewidths=1, levels=[6], zorder=103)
plt.clabel(con0, fontsize=8, fmt='%1.0f')
axes[0].set_title(f'(a) {storm_name} ERA5 ({date})')

for ax_idx, model in enumerate(models_to_plot, start=1):
    wsp_model = data[model]['leadtime']['wsp'].sel(lead_time=leadtime, time=date)
    slp_model = data[model]['leadtime']['z'].sel(lead_time=leadtime, time=date)
    q_model = data[model]['leadtime']['q'].sel(lead_time=leadtime, time=date)
    axes[ax_idx].contourf(wsp_model.lon, wsp_model.lat, wsp_model, cmap='plasma_r', levels=levels_wsp, extend='max', alpha=0.8, zorder=102)
    con1 = axes[ax_idx].contour(slp_model.lon, slp_model.lat, slp_model * factor_slp, colors='0.2', linestyles='solid', linewidths=0.8, levels=slp_levels, zorder=106)
    axes[ax_idx].contourf(q_model.lon, q_model.lat, q_model * 1000, cmap='PuBuGn', levels=levels_q850, extend='max', alpha=0.8, zorder=101)
    axes[ax_idx].contour(q_model.lon, q_model.lat, q_model * 1000, colors='b', linewidths=1, levels=[6], zorder=103)
    plt.clabel(con1, fontsize=8, fmt='%1.0f')
    axes[ax_idx].set_title(f'({letter[ax_idx]}) {model.upper()} forecast at {leadtime} hours')

rect = patches.Rectangle((box_lonW, box_latS), box_lonE - box_lonW, box_latN - box_latS, linewidth=2, edgecolor='magenta', facecolor='none', zorder=110)
axes[0].add_patch(rect)

f.subplots_adjust(hspace=0.05, wspace=0.05, left=0.0, right=1, bottom=0.05, top=0.95)
cbar1_ax = f.add_axes([0.05, 0.0, 0.4, 0.025])
cbar2_ax = f.add_axes([0.55, 0.0, 0.4, 0.025])
f.colorbar(cf0, cax=cbar1_ax, orientation='horizontal', label=wind_speed_label)
f.colorbar(cf1, cax=cbar2_ax, orientation='horizontal', label='Specific Humidity at 850hPa [g/kg]')

#plt.savefig(f'{storm_name}_lead_time_{leadtime}h_with_ifs_wsp850_q850_{date}_box.pdf', bbox_inches='tight')


In [ ]:
# WATER VAPOUR FLUX (q*wsp)

In [ ]:
# Same but for water vapour flux (q*wsp)

import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import numpy as np
import matplotlib.patches as patches

leadtime = 3 * 24
models_to_plot = ['fcnv2', 'pangu', 'ifs', 'aurora', 'aifs']
skip = 6
qv_color = 'r'
qv_scale = 2500
qv_min = 60
qv_width = 0.003
qv_head = 5

f, axes = plt.subplots(2, 3, figsize=(10, 7), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=0.0)))
axes = axes.flatten()

levels_wvf = np.arange(60, 200.1, 20)
slp_levels = np.arange(950, 1030, 5)
factor_slp = 0.01
letter = ['a', 'b', 'c', 'd', 'e', 'f']

for ax in axes:
    ax.coastlines(lw=1.0, color='green', zorder=105)
    ax.set_extent([lonW, lonE, latS, latN])

wvf_era = data['fcnv2']['leadtime']['wsp'].sel(lead_time=0, time=date) * data['fcnv2']['leadtime']['q'].sel(lead_time=0, time=date) * 1000
slp_era = data['fcnv2']['leadtime']['z'].sel(lead_time=0, time=date)
u_era = data['fcnv2']['leadtime']['u'].sel(lead_time=0, time=date)
v_era = data['fcnv2']['leadtime']['v'].sel(lead_time=0, time=date)
q_era = data['fcnv2']['leadtime']['q'].sel(lead_time=0, time=date) * 1000
cf0 = axes[0].contourf(wvf_era.lon, wvf_era.lat, wvf_era, cmap='YlGnBu', levels=levels_wvf, extend='max', alpha=0.8, zorder=102)
con0 = axes[0].contour(slp_era.lon, slp_era.lat, slp_era * factor_slp, colors='0.2', linestyles='solid', linewidths=0.8, levels=slp_levels, zorder=106)
wvf_x = np.ma.masked_where(wvf_era < qv_min, u_era.values * q_era.values)
wvf_y = np.ma.masked_where(wvf_era < qv_min, v_era.values * q_era.values)
axes[0].quiver(u_era.lon[::skip], u_era.lat[::skip], wvf_x[::skip, ::skip], wvf_y[::skip, ::skip], transform=ccrs.PlateCarree(), color=qv_color, scale=qv_scale, width=qv_width, headwidth=qv_head, zorder=108)
plt.clabel(con0, fontsize=8, fmt='%1.0f')
axes[0].set_title(f'(a) {storm_name} ERA5 ({date})')

for ax_idx, model in enumerate(models_to_plot, start=1):
    wvf_model = data[model]['leadtime']['wsp'].sel(lead_time=leadtime, time=date) * data[model]['leadtime']['q'].sel(lead_time=leadtime, time=date) * 1000
    slp_model = data[model]['leadtime']['z'].sel(lead_time=leadtime, time=date)
    u_model = data[model]['leadtime']['u'].sel(lead_time=leadtime, time=date)
    v_model = data[model]['leadtime']['v'].sel(lead_time=leadtime, time=date)
    q_model = data[model]['leadtime']['q'].sel(lead_time=leadtime, time=date) * 1000
    axes[ax_idx].contourf(wvf_model.lon, wvf_model.lat, wvf_model, cmap='YlGnBu', levels=levels_wvf, extend='max', alpha=0.8, zorder=102)
    con1 = axes[ax_idx].contour(slp_model.lon, slp_model.lat, slp_model * factor_slp, colors='0.2', linestyles='solid', linewidths=0.8, levels=slp_levels, zorder=106)
    plt.clabel(con1, fontsize=8, fmt='%1.0f')
    wvf_x = np.ma.masked_where(wvf_model < qv_min, u_model.values * q_model.values)
    wvf_y = np.ma.masked_where(wvf_model < qv_min, v_model.values * q_model.values)
    axes[ax_idx].quiver(u_model.lon[::skip], u_model.lat[::skip], wvf_x[::skip, ::skip], wvf_y[::skip, ::skip], transform=ccrs.PlateCarree(), color=qv_color, scale=qv_scale, width=qv_width, headwidth=qv_head, zorder=108)
    axes[ax_idx].set_title(f'({letter[ax_idx]}) {model.upper()} forecast at {leadtime} hours')

rect = patches.Rectangle((box_lonW, box_latS), box_lonE - box_lonW, box_latN - box_latS, linewidth=2, edgecolor='magenta', facecolor='none', zorder=110)
axes[0].add_patch(rect)

f.subplots_adjust(hspace=0.05, wspace=0.05, left=0.0, right=1, bottom=0.05, top=0.95)
cbar1_ax = f.add_axes([0.15, 0.0, 0.6, 0.025])
f.colorbar(cf0, cax=cbar1_ax, orientation='horizontal', label='water vapor flux (q*wsp) at 850hPa [g/kg * m/s]')

#plt.savefig(f'{storm_name}_lead_time_{leadtime}h_with_ifs_water_flux_{date}_box.pdf', bbox_inches='tight')


In [ ]:
# RMSE diagnostics against the ERA5-like reference (FCNv2 lead time 0)

import cartopy.crs as ccrs
import matplotlib.pyplot as plt

lead_window = slice(2 * 24, 5 * 24)
era_key = 'fcnv2'


def compute_rmse_field(model_key, var_key, lead_sel, unit_factor=1.0):
    model_field = data[model_key]['leadtime'][var_key].sel(lead_time=lead_sel, time=date) * unit_factor
    if model_key == 'ifs':
        ref = data['ifs']['leadtime'][var_key].sel(lead_time=0, time=date) * unit_factor
    else:
        ref = data[era_key]['leadtime'][var_key].sel(lead_time=0, time=date) * unit_factor
    diff = model_field - ref
    return np.sqrt((diff ** 2).mean(dim='lead_time'))




def compute_mb_field(model_key, var_key, lead_sel, unit_factor=1.0):
    model_field = data[model_key]['leadtime'][var_key].sel(lead_time=lead_sel, time=date) * unit_factor
    if model_key == 'ifs':
        ref = data['ifs']['leadtime'][var_key].sel(lead_time=0, time=date) * unit_factor
    else:
        ref = data[era_key]['leadtime'][var_key].sel(lead_time=0, time=date) * unit_factor
    diff = model_field - ref
    return diff.mean(dim='lead_time')

def compute_cyclone_center(reference_field):
    min_idx = np.unravel_index(np.nanargmin(reference_field.values), reference_field.shape)
    center_lat = float(reference_field.lat[min_idx[0]])
    center_lon = float(reference_field.lon[min_idx[1]])
    return center_lat, center_lon


In [ ]:
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

var_configs = [
    dict(var_key='z', unit_factor=0.01, levels=np.arange(1, 15, 1), cmap='Reds', colorbar_label='RMSE SLP [hPa]', contour_levels=np.arange(950, 1030, 5)),
    dict(var_key='wsp', unit_factor=1.0, levels=np.arange(1, 15, 1), cmap='gist_heat_r', colorbar_label=f'RMSE {wind_speed_label} [m/s]', contour_levels=get_wind_speed_levels(storm_name, level='aloft' if use_850_winds else 'surface')),
    dict(var_key='q', unit_factor=1000.0, levels=np.arange(0.5, 3.1, 0.25), cmap='YlGn', colorbar_label='RMSE q850 [g/kg]', contour_levels=np.arange(4, 9, 1)),
]

nrows = len(var_configs)
ncols = len(factual_models)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(3.6 * ncols, 3.3 * nrows),
    subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=0.0)),
)
axes = np.atleast_2d(axes)
colorbars = [None] * nrows

for row_idx, cfg in enumerate(var_configs):
    for col_idx, model in enumerate(factual_models):
        ax = axes[row_idx, col_idx]
        ax.coastlines(lw=0.8, color='0.3', zorder=10)
        ax.set_extent([lonW, lonE, latS, latN])

        rmse = compute_rmse_field(model, cfg['var_key'], lead_window, cfg['unit_factor'])
        lead_mean = data[model]['leadtime'][cfg['var_key']].sel(time=date, lead_time=lead_window).mean('lead_time') * cfg['unit_factor']

        cf = ax.contourf(rmse.lon, rmse.lat, rmse, cmap=cfg['cmap'], levels=cfg['levels'], extend='max', zorder=1)
        colorbars[row_idx] = colorbars[row_idx] or cf
        con = ax.contour(lead_mean.lon, lead_mean.lat, lead_mean, levels=cfg['contour_levels'], colors='0.25', linewidths=0.8, zorder=5)
        ax.clabel(con, fontsize=7, fmt='%1.0f')

        if row_idx == 0:
            ax.set_title(model.upper(), fontsize=11)
        if col_idx == 0:
            row_label = {
                'z': 'Sea level pressure',
                'wsp': wind_speed_label,
                'q': 'Specific humidity at 850 hPa',
            }[cfg['var_key']]
            ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90, va='center', ha='right', fontsize=11, fontweight='bold')

for row_idx, cf in enumerate(colorbars):
    cax = fig.add_axes([0.92, 0.11 + (nrows - 1 - row_idx) * 0.285, 0.015, 0.22])
    fig.colorbar(cf, cax=cax, orientation='vertical', label=var_configs[row_idx]['colorbar_label'])

fig.subplots_adjust(left=0.03, right=0.9, top=0.95, bottom=0.05, wspace=0.05, hspace=0.05)
rmse_map_filename = f'{storm_name}_rmse_multimodel_panel_{date}_ifs_analysis.pdf' if storm_name == 'Ciaran' else f'{storm_name}_rmse_multimodel_panel_{date}.pdf'
plt.savefig(rmse_map_filename, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

bias_configs = [
    dict(var_key='z', unit_factor=0.01, levels=np.arange(-9, 10, 2), cmap='RdBu_r', colorbar_label='MB SLP [hPa]', contour_levels=np.arange(950, 1020, 5)),
    dict(var_key='wsp', unit_factor=1.0, levels=np.arange(-11, 12, 2), cmap='RdBu_r', colorbar_label='MB Wind Speed [m/s]', contour_levels=np.arange(20, 46, 4)),
    dict(var_key='q', unit_factor=1000.0, levels=np.arange(-2.2, 2.4, 0.4), cmap='BrBG', colorbar_label='MB q850 [g/kg]', contour_levels=np.arange(4, 9, 2)),
]

nrows = len(bias_configs)
ncols = len(factual_models)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(0.5 + 3.2 * ncols, 3 * nrows),
    subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=0.0)),
)
axes = np.atleast_2d(axes)
cf_handles = [None] * nrows

for row_idx, cfg in enumerate(bias_configs):
    for col_idx, model in enumerate(factual_models):
        ax = axes[row_idx, col_idx]
        ax.coastlines(lw=0.8, color='green', zorder=5)
        ax.set_extent([lonW, lonE, latS, latN])

        mb = compute_mb_field(model, cfg['var_key'], lead_window, cfg['unit_factor'])
        lead_avg = data[model]['leadtime'][cfg['var_key']].sel(time=date, lead_time=lead_window).mean('lead_time') * cfg['unit_factor']

        cf = ax.contourf(mb.lon, mb.lat, mb, cmap=cfg['cmap'], levels=cfg['levels'], extend='both', zorder=1)
        if cf_handles[row_idx] is None:
            cf_handles[row_idx] = cf

        con = ax.contour(lead_avg.lon, lead_avg.lat, lead_avg, levels=cfg['contour_levels'], colors='0.3', linewidths=1, linestyles='solid', zorder=5)
        ax.clabel(con, fontsize=7, fmt='%1.0f')

        if row_idx == 0:
            ax.set_title(model.upper(), fontsize=11)
        if col_idx == 0:
            row_label = {
                'z': 'Sea level pressure',
                'wsp': 'Wind speed 850 hPa',
                'q': 'Specific humidity 850 hPa',
            }[cfg['var_key']]
            ax.text(-0.06, 0.5, row_label, transform=ax.transAxes, fontsize=12, fontweight='bold', va='center', ha='right', rotation=90)

for row_idx, cf in enumerate(cf_handles):
    cax = fig.add_axes([0.92, 0.08 + (nrows - 1 - row_idx) * 0.3, 0.015, 0.25])
    fig.colorbar(cf, cax=cax, orientation='vertical', label=bias_configs[row_idx]['colorbar_label'])

fig.subplots_adjust(left=0.02, right=0.9, top=0.95, bottom=0.05, wspace=0.05, hspace=0.05)
outname = f'{storm_name}_mb_multimodel_panel_{date}.pdf'
fig.savefig(outname, bbox_inches='tight')
print(f'Saved {outname}')
plt.show()


In [ ]:
import xarray as xr
from matplotlib.lines import Line2D
from pyproj import Geod

geod = Geod(ellps='WGS84')
window_points = 1
lead_grid = xr.DataArray(np.arange(0, 5.5 * 24 + 6, 6), dims='lead_time')


def geodesic_distance_grid(center_lat, center_lon, lat2d, lon2d):
    _, _, dist = geod.inv(
        np.full_like(lat2d, center_lon),
        np.full_like(lat2d, center_lat),
        lon2d,
        lat2d,
    )
    return dist


def rmse_area_series(model_key, var_key, unit_factor=1.0):
    pred = data[model_key]['leadtime'][var_key].sel(time=date) * unit_factor
    if model_key == 'ifs':
        ref = data['ifs']['leadtime'][var_key].sel(lead_time=0, time=date) * unit_factor
    else:
        ref = data[era_key]['leadtime'][var_key].sel(lead_time=0, time=date) * unit_factor
    diff = pred - ref
    rmse_list = []

    reference_slp = data[era_key]['leadtime']['z'].sel(lead_time=0, time=date)
    center_lat, center_lon = compute_cyclone_center(reference_slp)

    for lead in diff.lead_time:
        field = diff.sel(lead_time=lead)
        lon2d, lat2d = np.meshgrid(field.lon.values, field.lat.values)
        dist = geodesic_distance_grid(center_lat, center_lon, lat2d, lon2d)
        mask = xr.DataArray((dist <= validation_radius_m).astype(float), coords={'lat': field.lat, 'lon': field.lon}, dims=('lat', 'lon'))
        weights = xr.DataArray(np.cos(np.deg2rad(lat2d)), coords=mask.coords, dims=mask.dims)
        weighted_mask = mask * weights
        mse = ((field ** 2) * weighted_mask).sum(('lat', 'lon')) / weighted_mask.sum(('lat', 'lon'))
        rmse_list.append(np.sqrt(mse))

    return xr.concat(rmse_list, dim=diff.lead_time)


def rmse_box_series(model_key, var_key, unit_factor=1.0):
    pred = data[model_key]['leadtime'][var_key].sel(time=date, lat=slice(box_latN, box_latS), lon=slice(box_lonW, box_lonE)) * unit_factor
    if model_key == 'ifs':
        ref = data['ifs']['leadtime'][var_key].sel(lead_time=0, time=date, lat=slice(box_latN, box_latS), lon=slice(box_lonW, box_lonE)) * unit_factor
    else:
        ref = data[era_key]['leadtime'][var_key].sel(lead_time=0, time=date, lat=slice(box_latN, box_latS), lon=slice(box_lonW, box_lonE)) * unit_factor
    diff = pred - ref
    rmse_list = []

    for lead in diff.lead_time:
        field = diff.sel(lead_time=lead)
        lat_weights = np.cos(np.deg2rad(field.lat))
        weights = lat_weights.broadcast_like(field)
        mse = ((field ** 2) * weights).sum(('lat', 'lon')) / weights.sum(('lat', 'lon'))
        rmse_list.append(np.sqrt(mse))

    return xr.concat(rmse_list, dim=diff.lead_time)

if validation_geometry == 'circle':
    rmse_series = {
        'z': {model: rmse_area_series(model, 'z', 0.01) for model in factual_models},
        'wsp': {model: rmse_area_series(model, 'wsp', 1.0) for model in factual_models},
        'q': {model: rmse_area_series(model, 'q', 1000.0) for model in factual_models},
    }
    rmse_filename = f'{storm_name}_rmse_area_avg_timeseries_{date}_circle_{validation_radius_m // 1000}km_ifs_analysis.pdf'
else:
    rmse_series = {
        'z': {model: rmse_box_series(model, 'z', 0.01) for model in factual_models},
        'wsp': {model: rmse_box_series(model, 'wsp', 1.0) for model in factual_models},
        'q': {model: rmse_box_series(model, 'q', 1000.0) for model in factual_models},
    }
    rmse_filename = f'{storm_name}_rmse_area_avg_timeseries_{date}_box_ifs_vs_ana.pdf'

rmse_plot_config = [
    ('z', 'Area-avg RMSE SLP (hPa)'),
    ('wsp', f'Area-avg RMSE {wind_level_label} wind speed (m/s)'),
    ('q', 'Area-avg RMSE Q850 (g/kg)'),
]

fig, axes = plt.subplots(len(rmse_plot_config), 1, figsize=(6, 9), sharex=True)
model_colors = dict(zip(['fcnv2', 'pangu', 'aurora', 'aifs'], plt.cm.tab10.colors))
model_colors['ifs'] = 'k'
model_markers = dict(zip(['fcnv2', 'pangu', 'aurora', 'aifs'], ['o', 's', 'D', '^']))
model_markers['ifs'] = '*'

for ax, (var_key, ylabel) in zip(axes, rmse_plot_config):
    legend_handles = []
    for model in factual_models:
        series = rmse_series[var_key][model].rolling(lead_time=window_points, center=True, min_periods=1).mean().sel(lead_time=slice(None, 5.5 * 24))
        if model == 'ifs':
            smooth = series.interp(lead_time=lead_grid)[::2]
        else:
            smooth = series.reindex(lead_time=lead_grid, method='nearest', tolerance=1e-6)
        lead_days = -smooth['lead_time'] / 24.0
        score = float(smooth.sel(lead_time=slice(24, 120)).mean())
        color = model_colors[model]
        marker = model_markers[model]
        ax.plot(lead_days, smooth, color=color)
        ax.scatter(lead_days, smooth, color=color, marker=marker, s=30)
        legend_handles.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({score:.2f})', markersize=7))

    ax.axvspan(-5, -1, color='0.85', alpha=0.5, zorder=0)
    ax.set_ylabel(ylabel)
    ax.set_xlim(-5.5, 0)
    ax.grid(True, linestyle=':', color='0.7')
    ax.legend(handles=legend_handles, title='Model (1-5d avg)', ncol=2, loc='best')

axes[-1].set_xlabel('Lead time (days before event)')
plt.tight_layout()
plt.savefig(rmse_filename, bbox_inches='tight')
plt.show()


In [ ]:
# Cyclone-parameter comparison for the factual forecasts

from geopy.distance import geodesic

# Match the original Claudia notebook exactly: these diagnostics were
# evaluated on the full plotting domain, not on the AR box.
metric_lat_slice = slice(latN, latS)
metric_lon_slice = slice(lonW, lonE)

for model in factual_models:
    wsp_lag = data[model]['leadtime']['wsp'].sel(time=date, lat=metric_lat_slice, lon=metric_lon_slice)
    slp_lag = data[model]['leadtime']['z'].sel(time=date, lat=metric_lat_slice, lon=metric_lon_slice)
    q_lag = data[model]['leadtime']['q'].sel(time=date, lat=metric_lat_slice, lon=metric_lon_slice)

    data[model]['wsp_max'] = wsp_lag.max(('lat', 'lon'))
    data[model]['q850_max'] = q_lag.max(('lat', 'lon'))
    data[model]['slp_min'] = slp_lag.min(('lat', 'lon'))

# Match the original notebook exactly: low-center position was computed
# from the full leadtime field at the event time, not from the plotting box.
ref_z = data[era_key]['leadtime']['z'].sel(time=date)
ref_min_idx = ref_z.isel(lead_time=0).argmin(dim=['lat', 'lon'])
ref_lat = float(ref_z['lat'][ref_min_idx['lat']])
ref_lon = float(ref_z['lon'][ref_min_idx['lon']])
ref_coord = (ref_lat, ref_lon)
DIST_THRESHOLD_KM = 1000

for model in factual_models:
    z_model = data[model]['leadtime']['z'].sel(time=date)
    error_km = []

    for lt in range(len(z_model['lead_time'])):
        z_slice = z_model.isel(lead_time=lt)
        if np.isnan(z_slice).all():
            error_km.append(np.nan)
            continue

        min_idx = z_slice.argmin(dim=['lat', 'lon'])
        lat = float(z_slice['lat'][min_idx['lat']])
        lon = float(z_slice['lon'][min_idx['lon']])
        dist = geodesic((lat, lon), ref_coord).kilometers
        error_km.append(dist if dist < DIST_THRESHOLD_KM else np.nan)

    data[model]['location_error_km'] = xr.DataArray(error_km, coords=[z_model['lead_time']], dims=['lead_time'])


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

window_points = 3
avg_slice = slice(24, 5 * 24)
lead_grid = xr.DataArray(np.arange(0, 6 * 24 + 6, 6), dims='lead_time')

fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)
model_colors = dict(zip(['fcnv2', 'pangu', 'aurora', 'aifs'], plt.cm.tab10.colors))
model_colors['ifs'] = 'k'
model_markers = dict(zip(['fcnv2', 'pangu', 'aurora', 'aifs'], ['o', 's', 'D', '^']))
model_markers['ifs'] = 'v'
legend_handles_1 = []
legend_handles_2 = []
legend_handles_3 = []

era_wsp = data[era_key]['wsp_max'].sel(lead_time=0)
era_q = data[era_key]['q850_max'].sel(lead_time=0) * 1000.0
era_slp = (data[era_key]['slp_min'] * 0.01).sel(lead_time=0)
ifsana_wsp = data['ifs']['wsp_max'].sel(lead_time=0)
ifsana_q = data['ifs']['q850_max'].sel(lead_time=0) * 1000.0
ifsana_slp = (data['ifs']['slp_min'] * 0.01).sel(lead_time=0)


def smooth_align(series, model):
    smoothed = series.rolling(lead_time=window_points, center=True, min_periods=1).mean()
    if model == 'ifs':
        return smoothed.interp(lead_time=lead_grid)
    return smoothed.reindex(lead_time=lead_grid, method='nearest', tolerance=1e-6)

for model in factual_models:
    wsp = data[model]['wsp_max']
    q = data[model]['q850_max'] * 1000.0
    slp = data[model]['slp_min'] * 0.01
    loc_error = data[model]['location_error_km']

    wsp_smooth = smooth_align(wsp, model)
    q_smooth = smooth_align(q, model)
    slp_smooth = smooth_align(slp, model)
    loc_smooth = smooth_align(loc_error, model)
    lead_days = -wsp_smooth['lead_time'] / 24

    if model == 'ifs':
        wsp_diff = float((wsp_smooth - ifsana_wsp).sel(lead_time=avg_slice).mean())
        q_diff = float((q_smooth - ifsana_q).sel(lead_time=avg_slice).mean())
        slp_diff = float((slp_smooth - ifsana_slp).sel(lead_time=avg_slice).mean())
        loc_diff = float(loc_smooth.sel(lead_time=avg_slice).mean())
    else:
        wsp_diff = float((wsp_smooth - era_wsp).sel(lead_time=avg_slice).mean())
        q_diff = float((q_smooth - era_q).sel(lead_time=avg_slice).mean())
        slp_diff = float((slp_smooth - era_slp).sel(lead_time=avg_slice).mean())
        loc_diff = float(loc_smooth.sel(lead_time=avg_slice).mean())

    color = model_colors[model]
    marker = model_markers[model]
    scatter_slice = slice(None, None, 2) if model == 'ifs' else slice(None)

    if storm_name == 'Claudia':
        axes[0].plot(lead_days, wsp_smooth, color=color)
        axes[0].scatter(lead_days.values[scatter_slice], wsp_smooth.values[scatter_slice], s=25, marker=marker, color=color)
        axes[1].plot(lead_days, q_smooth, color=color)
        axes[1].scatter(lead_days.values[scatter_slice], q_smooth.values[scatter_slice], s=25, marker=marker, color=color)
        axes[2].plot(lead_days, loc_smooth, color=color)
        axes[2].scatter(lead_days.values[scatter_slice], loc_smooth.values[scatter_slice], s=25, marker=marker, color=color)

        legend_handles_1.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({wsp_diff:+.1f})', markersize=10))
        legend_handles_2.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({q_diff:+.1f})', markersize=10))
        legend_handles_3.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({loc_diff:+.0f})', markersize=10))
    else:
        axes[0].plot(lead_days, slp_smooth, color=color)
        axes[0].scatter(lead_days.values[scatter_slice], slp_smooth.values[scatter_slice], s=25, marker=marker, color=color)
        axes[1].plot(lead_days, wsp_smooth, color=color)
        axes[1].scatter(lead_days.values[scatter_slice], wsp_smooth.values[scatter_slice], s=25, marker=marker, color=color)
        axes[2].plot(lead_days, loc_smooth, color=color)
        axes[2].scatter(lead_days.values[scatter_slice], loc_smooth.values[scatter_slice], s=25, marker=marker, color=color)

        legend_handles_1.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({slp_diff:+.1f})', markersize=10))
        legend_handles_2.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({wsp_diff:+.1f})', markersize=10))
        legend_handles_3.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=f'{model.upper()} ({loc_diff:+.0f})', markersize=10))

if storm_name == 'Claudia':
    axes[0].axhline(float(era_wsp), color='black', linestyle='--')
    axes[1].axhline(float(era_q), color='black', linestyle='--')
    axes[0].axhline(float(ifsana_wsp), color='grey', linestyle=':')
    axes[1].axhline(float(ifsana_q), color='grey', linestyle=':')
    axes[0].set_ylabel('Max WSP850 (m/s)', fontsize=12)
    axes[1].set_ylabel('Max Q850 (g/kg)', fontsize=12)
    axes[2].set_ylabel('Low center position error (km)', fontsize=12)
else:
    axes[0].axhline(float(era_slp), color='black', linestyle='--')
    axes[1].axhline(float(era_wsp), color='black', linestyle='--')
    axes[0].axhline(float(ifsana_slp), color='grey', linestyle=':')
    axes[1].axhline(float(ifsana_wsp), color='grey', linestyle=':')
    axes[0].set_ylabel('Minimum SLP (hPa)', fontsize=12)
    axes[1].set_ylabel('Max WSP850 (m/s)', fontsize=12)
    axes[2].set_ylabel('Low center position error (km)', fontsize=12)
    axes[0].set_ylim([940, 975])
    axes[1].set_ylim([30, 55])
    axes[2].set_ylim([0, 600])

for ax in axes:
    ax.grid(True, linestyle=':', color='gray', alpha=0.6)
    ax.axvspan(-5, -1, color='lightgray', alpha=0.4, zorder=0)
    ax.tick_params(labelsize=10)
    ax.set_xlim(-5.5, 0)

axes[2].set_xlabel('Forecast lead time (days before event)', fontsize=12)
axes[2].xaxis.set_major_locator(mticker.MultipleLocator(1))
axes[2].xaxis.set_minor_locator(mticker.AutoMinorLocator())

legend_title = 'Model - reference (avg 1-5 d)'
axes[0].legend(handles=legend_handles_1, title=legend_title, ncol=2, loc='best')
axes[1].legend(handles=legend_handles_2, title=legend_title, ncol=2, loc='best')
axes[2].legend(handles=legend_handles_3, title=legend_title, ncol=2, loc='best')

fig.subplots_adjust(hspace=0.0, wspace=0.02)
plt.tight_layout(pad=0.8)
plt.savefig(f'{storm_name}_low_parameters_timeseries_{date}_ifs_vs_ana.pdf', bbox_inches='tight')
plt.show()


# Attribution

In [ ]:
#let's do the same we did for the factual
init_times = []
current_time = t0_i
while current_time <= t0_f:
    init_times.append(current_time)
    current_time += datetime.timedelta(hours=delta_h)

In [ ]:
# Load counterfactual forecasts for the four paper models
models = ['fcnv2', 'pangu', 'aurora', 'aifs']
data_cf = {}

for model in models:
    if model in ['fcnv2', 'pangu']:
        root = EARTH2MIP_FORECAST_ROOT
        u_var = 'u850' if use_850_winds else 'u10m'
        v_var = 'v850' if use_850_winds else 'v10m'
        data_cf[model] = {
            'u': utils.load_data(u_var, init_times, root, extension=PGW_EXTENSION, model=model),
            'v': utils.load_data(v_var, init_times, root, extension=PGW_EXTENSION, model=model),
            'z': utils.load_data('msl', init_times, root, extension=PGW_EXTENSION, model=model),
        }
        if model == 'pangu':
            data_cf[model]['q'] = utils.load_data('q850', init_times, root, extension=PGW_EXTENSION, model=model)
        else:
            r = utils.load_data('r850', init_times, root, extension=PGW_EXTENSION, model='fcnv2')
            t = utils.load_data('t850', init_times, root, extension=PGW_EXTENSION, model='fcnv2')
            data_cf[model]['q'] = relative_humidity_to_specific_humidity(r, t, 850)
    elif model == 'aurora':
        root = AURORA_FORECAST_ROOT
        level = 850 if use_850_winds else None
        u_var = 'u' if use_850_winds else '10u'
        v_var = 'v' if use_850_winds else '10v'
        data_cf[model] = {
            'u': utils.load_data(u_var, init_times, root, level=level, extension=PGW_EXTENSION, model=model),
            'v': utils.load_data(v_var, init_times, root, level=level, extension=PGW_EXTENSION, model=model),
            'z': utils.load_data('msl', init_times, root, level=None, extension=PGW_EXTENSION, model=model),
            'q': utils.load_data('q', init_times, root, level=850, extension=PGW_EXTENSION, model=model),
        }
    elif model == 'aifs':
        root = AIFS_FORECAST_ROOT
        u_var = 'u_850' if use_850_winds else '10u'
        v_var = 'v_850' if use_850_winds else '10v'
        data_cf[model] = {
            'u': utils.load_data(u_var, init_times, root, extension=PGW_EXTENSION, model=model),
            'v': utils.load_data(v_var, init_times, root, extension=PGW_EXTENSION, model=model),
            'z': utils.load_data('msl', init_times, root, extension=PGW_EXTENSION, model=model),
            'q': utils.load_data('q_850', init_times, root, extension=PGW_EXTENSION, model=model),
        }


In [ ]:
# Area-sliced counterfactual data structure

lat_slice = slice(latN, latS)
lon_slice = slice(lonW, lonE)

persist_items_cf = []
for model in models:
    d = data_cf[model]
    data_cf[model]['area'] = {
        'u': utils.subset_region(d['u'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
        'v': utils.subset_region(d['v'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
        'z': utils.subset_region(d['z'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
        'q': utils.subset_region(d['q'], lat_slice, lon_slice).sel(time=time_slice).transpose('init_time', 'time', 'lat', 'lon'),
    }
    for key, value in data_cf[model]['area'].items():
        persist_items_cf.append((model, key, value))

persisted_vars_cf = persist(*(value for _, _, value in persist_items_cf))
for (model, key, _), value in zip(persist_items_cf, persisted_vars_cf):
    data_cf[model]['area'][key] = value


In [ ]:
for model in models:
    area_data_cf = data_cf[model]['area']
    
    # Calculate wind speed
    u = area_data_cf['u']
    v = area_data_cf['v']
    wsp = np.sqrt(u**2 + v**2)
    
    # Select valid time range
    time_range = wsp.time.sel(time=time_slice)

    # Convert init_time to lead_time
    wsp_lag = utils.from_init_time_to_leadtime(wsp, init_time_min, lead_time_range, time_range)
    u_lag = utils.from_init_time_to_leadtime(area_data_cf['u'], init_time_min, lead_time_range, time_range)
    v_lag = utils.from_init_time_to_leadtime(area_data_cf['v'], init_time_min, lead_time_range, time_range)
    z_lag = utils.from_init_time_to_leadtime(area_data_cf['z'], init_time_min, lead_time_range, time_range)
    q_lag = utils.from_init_time_to_leadtime(area_data_cf['q'], init_time_min, lead_time_range, time_range)
    

    # Save to dictionary
    data_cf[model]['leadtime'] = {
        'wsp': wsp_lag,
        'u': u_lag,
        'v': v_lag,
        'z': z_lag,
        'q': q_lag,
        # Add others like 'tcwv' or 'q' if needed later
    }

In [ ]:
#diff between factual and counterfactual (ACC signal)
for model in models:
    data[model]['acc_signal_leadtime'] = {}
    for var in ['wsp','z','q']:
        data[model]['acc_signal_leadtime'][var] = data[model]['leadtime'][var] - data_cf[model]['leadtime'][var]

comment:
the signal is quite sensititive to the leadtime chosen


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import string
from scipy.stats import ttest_1samp
from scipy.stats import ttest_ind

# Create subplots with linked axes
f, axes = plt.subplots(3,4, sharey=True, sharex=True, figsize=(14,7.8), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=0.0)),
                      gridspec_kw={'wspace': 0.1, 'hspace': 0.1})
axes= axes.flatten()
lt1,lt2=int(2*24),int(5*24)
lead_time_S= slice(lt1,lt2)

for ax in axes:
    ax.set_aspect(1)
    ax.coastlines(lw=1.0, color='k', zorder=105)
    ax.set_extent([lonW, lonE, latS, latN])
    #ax.axvline(x=0, color='0.5', linestyle='--')
    #ax.axhline(y=0, color='0.5', linestyle='--')

for i, model in enumerate(models):
    axes[0+i].set_title(model.capitalize())

    # --- SLP change ---
    z_factual = data[model]['leadtime']['z'].sel(lead_time=lead_time_S, time=date)
    z_counterfactual = data_cf[model]['leadtime']['z'].sel(lead_time=lead_time_S, time=date)
    z_plot = (z_factual.mean('lead_time') - z_counterfactual.mean('lead_time')) * 0.01
    z_contours = z_factual.mean('lead_time') * 0.01
    contours_z = np.arange(940, 1035, 5)

    # --- Two-sample t-test ---
    tstat, pvals = ttest_ind(
        z_factual.values, z_counterfactual.values,
        axis=0, nan_policy='omit', equal_var=False
    )
    sig_mask = (pvals < 0.05)
    nonsig_mask = ~sig_mask

    lon2d, lat2d = np.meshgrid(z_plot.lon, z_plot.lat)
    step = 2
    sub_mask = np.zeros_like(nonsig_mask, dtype=bool)
    sub_mask[::step, ::step] = nonsig_mask[::step, ::step]
    axes[0+i].scatter(
        lon2d[sub_mask], lat2d[sub_mask],
        color='0.5', s=2, marker='.', alpha=0.7, zorder=130
    )
    cf0 = axes[0+i].contourf(z_plot.lon, z_plot.lat, z_plot, cmap='RdBu_r', levels=np.arange(-5, 5.1, 0.5), extend='both')
    con0 = axes[0+i].contour(z_contours.lon, z_contours.lat, z_contours, colors=['g'], linewidths=0.8, levels=contours_z, zorder=106)

    # --- WSP ---
    wsp_factual = data[model]['leadtime']['wsp'].sel(lead_time=lead_time_S, time=date)
    wsp_counterfactual = data_cf[model]['leadtime']['wsp'].sel(lead_time=lead_time_S, time=date)
    wsp_plot = (wsp_factual.mean('lead_time') - wsp_counterfactual.mean('lead_time'))
    tstat, pvals = ttest_ind(
        wsp_factual.values, wsp_counterfactual.values,
        axis=0, nan_policy='omit', equal_var=False
    )
    sig_mask = (pvals < 0.05)
    nonsig_mask = ~sig_mask
    lon2d, lat2d = np.meshgrid(wsp_plot.lon, wsp_plot.lat)
    sub_mask = np.zeros_like(nonsig_mask, dtype=bool)
    sub_mask[::step, ::step] = nonsig_mask[::step, ::step]
    axes[4+i].scatter(
        lon2d[sub_mask], lat2d[sub_mask],
        color='0.5', s=2, marker='.', alpha=1, zorder=130
    )
    cf1 = axes[4+i].contourf(wsp_plot.lon, wsp_plot.lat, wsp_plot, cmap='RdGy_r', levels=np.arange(-5, 5.1, 0.5), extend='both')
    con1 = axes[4+i].contour(z_contours.lon, z_contours.lat, z_contours, colors=['g'], linewidths=0.8, levels=contours_z, zorder=106)

    # --- Q ---
    q_factual = data[model]['leadtime']['q'].sel(lead_time=lead_time_S, time=date) * 1000
    q_counterfactual = data_cf[model]['leadtime']['q'].sel(lead_time=lead_time_S, time=date) * 1000
    q_plot = (q_factual.mean('lead_time') - q_counterfactual.mean('lead_time'))
    tstat, pvals = ttest_ind(
        q_factual.values, q_counterfactual.values,
        axis=0, nan_policy='omit', equal_var=False
    )
    sig_mask = (pvals < 0.05)
    nonsig_mask = ~sig_mask
    lon2d, lat2d = np.meshgrid(q_plot.lon, q_plot.lat)
    sub_mask = np.zeros_like(nonsig_mask, dtype=bool)
    sub_mask[::step, ::step] = nonsig_mask[::step, ::step]
    axes[8+i].scatter(
        lon2d[sub_mask], lat2d[sub_mask],
        color='0.5', s=1, marker='.', alpha=1, zorder=130
    )
    cf2 = axes[8+i].contourf(q_plot.lon, q_plot.lat, q_plot, cmap='BrBG', levels=np.arange(-1.5, 1.6, 0.25), extend='both')
    con2 = axes[8+i].contour(z_contours.lon, z_contours.lat, z_contours, colors=['g'], linewidths=0.8, levels=contours_z, zorder=106)


# Add variable name labels to the first column of each row
axes[0].text(0.02, 1.12, 'Sea Level Pressure', transform=axes[0].transAxes,
             fontsize=12, fontweight='bold', va='bottom', ha='left')

axes[4].text(0.02, 1.02, 'Wind Speed at 850 hPa', transform=axes[4].transAxes,
             fontsize=12, fontweight='bold', va='bottom', ha='left')

axes[8].text(0.02, 1.02, 'Specific Humidity at 850 hPa', transform=axes[8].transAxes,
             fontsize=12, fontweight='bold', va='bottom', ha='left')

# Add subplot labels (a), (b), (c), ...
for idx, ax in enumerate(axes):
    label = f'{string.ascii_lowercase[idx]}'
    ax.text(0.02, 0.98, label, transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top', ha='left',
            bbox=dict(facecolor='white', edgecolor='none', pad=2.5, alpha=1), zorder=200)


f.subplots_adjust(hspace=0.04, wspace=0.0, left=0.0, right=0.98, bottom=0.02, top=0.95)


cbar0 = plt.colorbar(cf0, ax=axes[0:4], label='SLP difference [hPa]', pad=0.02, orientation='vertical', shrink=0.95, aspect=15)
cbar1 = plt.colorbar(cf1, ax=axes[4:8], label='WSP850 difference [m/s]', pad=0.02, orientation='vertical', shrink=0.95, aspect=15)
cbar2 = plt.colorbar(cf2, ax=axes[8:12], label=r'Q850 difference [$g/kg$]', pad=0.02, orientation='vertical', shrink=0.95, aspect=15)



plt.savefig(f'extratropical_storm_{storm_name}_PGW_attribution_mapplot_slp_wsp850_q850_multimodel_leadtimes_{lt1}h_to_{lt2}h_{date}.pdf',bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap

def spectral_white_center(name='SpectralWhiteCenter', n=256, white_width=0.05):
    """
    Spectral-like diverging colormap with a white band in the center.
    white_width: fraction of colormap reserved for white around 0.5 (e.g. 0.08-0.15).
    """
    base = mpl.cm.get_cmap('Spectral')  # or 'Spectral' if you prefer opposite direction
    x = np.linspace(0, 1, n)
    colors = base(x)

    w = max(0.0, min(0.45, white_width / 2.0))
    left = 0.5 - w
    right = 0.5 + w

    # indices of white center band
    mask = (x >= left) & (x <= right)
    colors[mask, :3] = 1.0  # RGB -> white
    colors[mask, 3] = 1.0   # alpha

    return LinearSegmentedColormap.from_list(name, colors, N=n)

# use
cmap = spectral_white_center(white_width=0.05)

# example in contourf:
# cf = ax.contourf(lon, lat, field, levels=levels, cmap=cmap, extend='both')


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
from scipy.stats import ttest_1samp
import string

# Lead-time window
lt1, lt2 = int(2 * 24), int(5 * 24)
lead_time_S = slice(lt1, lt2)

nmod = len(models)
fig, axes = plt.subplots(
    4, nmod, figsize=(3 * nmod, 13),
    subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=0.0)),
    gridspec_kw={'wspace': 0.08, 'hspace': 0.12}
)
if nmod == 1:
    axes = np.array(axes).reshape(3, 1)

levels_wvf = np.arange(-45, 45.1, 2.5)   # misma escala para las 3 filas
contours_z = np.arange(940, 1035, 5)
step = 2

row_titles = [
    r'$\Delta Q \cdot WSP$',
    r'$Q \cdot \Delta WSP$ ',
    r'$\Delta Q \cdot \Delta WSP$',
    r'$\Delta( Q \cdot WSP)$'
]

cf_last = None

for i, model in enumerate(models):
    # Campos factual / counterfactual
    q_f = data[model]['leadtime']['q'].sel(lead_time=lead_time_S, time=date)
    q_c = data_cf[model]['leadtime']['q'].sel(lead_time=lead_time_S, time=date)
    w_f = data[model]['leadtime']['wsp'].sel(lead_time=lead_time_S, time=date)
    w_c = data_cf[model]['leadtime']['wsp'].sel(lead_time=lead_time_S, time=date)

    dq = q_f - q_c
    dw = w_f - w_c

    # Términos por lead_time (para test y media)
    term1_lt = dq * w_f * 1000.0
    term2_lt = q_f * dw * 1000.0
    term3_lt = dq * dw * 1000.0
    term4_lt = (q_f * w_f - q_c * w_c) * 1000.0  # "como está"

    term_lts = [term1_lt, term2_lt, term3_lt, term4_lt]

    # SLP factual media (contornos)
    z_contours = data[model]['leadtime']['z'].sel(lead_time=lead_time_S, time=date).mean('lead_time') * 0.01

    for r in range(4):
        ax = axes[r, i]
        ax.set_aspect(1)
        ax.coastlines(lw=1.0, color='k', zorder=105)
        ax.set_extent([lonW, lonE, latS, latN])

        plot_field = term_lts[r].mean('lead_time')

        # Test contra 0 a lo largo de lead_time (stipple en no-significativo)
        _, pvals = ttest_1samp(term_lts[r].values, popmean=0.0, axis=0, nan_policy='omit')
        nonsig_mask = ~(pvals < 0.05)

        lon2d, lat2d = np.meshgrid(plot_field.lon, plot_field.lat)
        sub_mask = np.zeros_like(nonsig_mask, dtype=bool)
        sub_mask[::step, ::step] = nonsig_mask[::step, ::step]
        ax.scatter(lon2d[sub_mask], lat2d[sub_mask], color='0.5', s=2, marker='.', alpha=0.8, zorder=130)

        cf_last = ax.contourf(
            plot_field.lon, plot_field.lat, plot_field,
            cmap=cmap, levels=levels_wvf, extend='both'
        )
        ax.contour(
            z_contours.lon, z_contours.lat, z_contours,
            colors='g', linewidths=0.8, levels=contours_z, zorder=106
        )

        # Títulos por fila/columna
        if r == 0:
            ax.set_title(model.capitalize())

        if i == 0:
            ax.text(
                0.02, 1.0, row_titles[r], transform=ax.transAxes,
                fontsize=12, fontweight='bold', va='bottom', ha='left'
            )

        # panel label
        panel_idx = r * nmod + i
        ax.text(
            0.02, 0.98, f'{string.ascii_lowercase[panel_idx]}', transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top', ha='left',
            bbox=dict(facecolor='white', edgecolor='none', pad=2.5, alpha=1), zorder=200
        )

fig.subplots_adjust(left=0.03, right=0.92, bottom=0.03, top=0.95, wspace=0.05, hspace=0.12)
cbar = plt.colorbar(cf_last, ax=axes.ravel().tolist(), orientation='horizontal', pad=0.02, shrink=0.7, aspect=30)
cbar.set_label(r'[g kg$^{-1}$ m s$^{-1}$]')
cbar.set_ticks(np.arange(-40, 40.1, 10))  # set ticks at the same levels as the contourf



plt.savefig(f'extratropical_storm_{storm_name}_PGW_attribution_mapplot_wvf_multimodel_leadtimes_{lt1}h_to_{lt2}h_{date}.pdf', bbox_inches='tight')



In [ ]:

# Calculate maxima/minima within the event box for factual and counterfactual runs
for dataset in [data, data_cf]:
    for model in models:
        wsp_lag = dataset[model]['leadtime']['wsp'].sel(time=date)
        slp_lag = dataset[model]['leadtime']['z'].sel(time=date)
        dataset[model]['wsp_max'] = wsp_lag.max(['lat', 'lon'])
        dataset[model]['slp_min'] = slp_lag.min(['lat', 'lon'])

In [ ]:
from geopy.distance import geodesic

# Reference location from ERA5 (fcnv2 at lead_time = 0)
ref_z = data_cf['fcnv2']['leadtime']['z'].sel(time=date, lat=lat_slice, lon=lon_slice)
ref_min_idx = ref_z.isel(lead_time=0).argmin(dim=["lat", "lon"])
ref_lat = float(ref_z['lat'][ref_min_idx['lat']])
ref_lon = float(ref_z['lon'][ref_min_idx['lon']])
ref_coord = (ref_lat, ref_lon)

# Distance threshold in km
DIST_THRESHOLD_KM = 500

for model in models:
    z_model = data[model]['leadtime']['z'].sel(time=date, lat=lat_slice, lon=lon_slice)
    z_model_cf = data_cf[model]['leadtime']['z'].sel(time=date, lat=lat_slice, lon=lon_slice)

    error_km = []

    for lt in range(len(z_model['lead_time'])):
        z_slice = z_model.isel(lead_time=lt)
        z_slice_cf = z_model_cf.isel(lead_time=lt)

        # Skip lead times with NaNs
        if np.isnan(z_slice).all():
            error_km.append(np.nan)
            continue

        min_idx = z_slice.argmin(dim=["lat", "lon"])
        lat = float(z_slice['lat'][min_idx['lat']])
        lon = float(z_slice['lon'][min_idx['lon']])
        model_coord = (lat, lon)

        min_idx_cf = z_slice_cf.argmin(dim=["lat", "lon"])
        lat = float(z_slice_cf['lat'][min_idx_cf['lat']])
        lon = float(z_slice_cf['lon'][min_idx_cf['lon']])
        model_coord_cf = (lat, lon)

        # Compute distance to reference
        dist = geodesic(model_coord, model_coord_cf).kilometers

        if dist < DIST_THRESHOLD_KM:
            error_km.append(dist)
        else:
            error_km.append(np.nan)

    data_cf[model]['location_diff'] = xr.DataArray(
        error_km,
        coords=[z_model['lead_time']],
        dims=["lead_time"]
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
import xarray as xr

fig, axes = plt.subplots(3, 1, figsize=(6, 9), sharex=True)

markers = ['o', 's', 'D', '^', 'v', 'P', '*', 'X', '<', '>']
colors = plt.cm.tab10.colors
model_markers = {model: markers[i % len(markers)] for i, model in enumerate(models)}
model_colors = {model: colors[i % len(colors)] for i, model in enumerate(models)}
legend_handles = []

mm_wsp, mm_slp, mm_loc = [], [], []

for model in models:
    wsp_diff = (data[model]['wsp_max'] - data_cf[model]['wsp_max']).rolling(lead_time=4, center=True, min_periods=1).mean()
    slp_diff = (data[model]['slp_min'] - data_cf[model]['slp_min']).rolling(lead_time=4, center=True, min_periods=1).mean() * 0.01
    loc_diff = data_cf[model].get('location_diff')
    if loc_diff is not None:
        loc_diff = loc_diff.rolling(lead_time=4, center=True, min_periods=1).mean()

    lead_days = -wsp_diff['lead_time'] / 24
    marker = model_markers[model]
    color = model_colors[model]
    label = model.capitalize()

    axes[0].plot(lead_days, wsp_diff, color=color, alpha=0.8)
    axes[0].scatter(lead_days, wsp_diff, s=25, marker=marker, color=color)

    axes[1].plot(lead_days, slp_diff, color=color, alpha=0.8)
    axes[1].scatter(lead_days, slp_diff, s=25, marker=marker, color=color)

    if loc_diff is not None:
        axes[2].plot(lead_days, loc_diff, color=color, alpha=0.8)
        axes[2].scatter(lead_days, loc_diff, s=25, marker=marker, color=color)

    legend_handles.append(Line2D([0], [0], color=color, marker=marker, linestyle='-', label=label, markersize=10))

    mm_wsp.append(wsp_diff)
    mm_slp.append(slp_diff)
    if loc_diff is not None:
        mm_loc.append(loc_diff)

if mm_wsp:
    wsp_mean = xr.concat(mm_wsp, dim='model').mean('model')
    slp_mean = xr.concat(mm_slp, dim='model').mean('model')
    lead_days_mm = -wsp_mean['lead_time'] / 24

    axes[0].plot(lead_days_mm, wsp_mean, color='black', linewidth=2.2, label='Multimodel')
    axes[1].plot(lead_days_mm, slp_mean, color='black', linewidth=2.2)
    legend_handles.append(Line2D([0], [0], color='black', linewidth=2.2, label='Multimodel'))

if mm_loc:
    loc_mean = xr.concat(mm_loc, dim='model').mean('model')
    axes[2].plot(lead_days_mm, loc_mean, color='black', linewidth=2.2)

for ax in axes:
    ax.axhline(0, color='black', linestyle='--', lw=1)

legend_handles.append(Line2D([0], [0], color='black', linestyle='--', label='No change'))

ylabels = ['Δ Max 850hPa WSP (m/s)', 'Δ Min SLP (hPa)', 'Δ Low position (km)']
for ax, ylabel in zip(axes, ylabels):
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, linestyle=':', color='gray', alpha=0.6)
    ax.axvspan(-5, -2, color='lightgray', alpha=0.4, zorder=0)
    ax.tick_params(labelsize=10)
    ax.set_xlim(-6, 0)

axes[0].set_ylim(-5, 5)
axes[1].set_ylim(-8, 8)
axes[2].set_ylim(0, 400)

axes[2].set_xlabel('Forecast lead time (days before event)', fontsize=12)
axes[2].xaxis.set_major_locator(mticker.MultipleLocator(1))
axes[2].xaxis.set_minor_locator(mticker.AutoMinorLocator())

fig.subplots_adjust(hspace=0.0, wspace=0.02)
plt.tight_layout(pad=0.8)

axes[0].legend(handles=legend_handles, title='Model', ncol=2, fontsize=9, title_fontsize=10)

plt.savefig(f'extratropical_storm_{storm_name}_leadtime_vs_differences_attribution_{date}_smoothed.pdf', bbox_inches='tight')
plt.show()

